In [49]:
# Standard library
import os
import random
import time
from pathlib import Path

# Data
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import (
    Dataset,
    DataLoader
)

# Torchvision
import torchvision
import torchvision.transforms as transforms

# Progress bars
from tqdm import tqdm

# Experiment tracking
import wandb


plt.style.use("default")
import sys
from dataclasses import dataclass

In [50]:
# 1. Clone your clean repository code from GitHub
REPO_URL = "https://github.com/K0NSTANT1N3/Facial-Expression-Recognition.git"
REPO_NAME = "Facial-Expression-Recognition"

if not os.path.exists(REPO_NAME):
    print("Cloning repository...")
    os.system(f"git clone {REPO_URL}")
else:
    print("Repository already exists. Pulling latest updates...")
    os.system(f"cd {REPO_NAME} && git pull")

# 2. THE SECRET SAUCE: Add your cloned repo directory to Python's path
sys.path.append(os.path.abspath(REPO_NAME))

print("Successfully linked GitHub repository and adjusted system paths!")


Repository already exists. Pulling latest updates...
Already up to date.
Successfully linked GitHub repository and adjusted system paths!


In [51]:
import wandb

wandb.login(key="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")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


True

In [52]:
@dataclass
class Config:
    csv_path: str = "/kaggle/input/competitions/challenges-in-representation-learning-facial-expression-recognition-challenge/train.csv"

    batch_size: int = 64
    lr: float = 1e-3
    epochs: int = 20

    num_classes: int = 7
    image_size: int = 48

    random_seed: int = 67


config = Config()

In [53]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(config.random_seed)

In [54]:
sweep_config = {
    "method": "random",

    "metric": {
        "name": "val_acc",
        "goal": "maximize"
    },

    "parameters": {
        "lr": {
            "values": [1e-2, 1e-3, 5e-4, 1e-4]
        },

        "dropout": {
            "values": [0.2, 0.3, 0.5]
        },

        "batch_size": {
            "values": [32, 64, 128]
        },
        "epochs": {
            "value": 20
        }
    }
}

In [55]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)

cuda


In [56]:
df = pd.read_csv(config.csv_path)

print(df.shape)
df.head()

(28709, 2)


,emotion,pixels
0,0,70 80 82 72 58 58 60 63 54 58 60 48 89 115 121...
1,0,151 150 147 155 148 133 111 140 170 174 182 15...
2,2,231 212 156 164 174 138 161 173 182 200 106 38...
3,4,24 32 36 30 32 23 19 20 30 41 21 22 32 34 21 1...
4,6,4 0 0 0 0 0 0 0 0 0 0 0 3 15 23 28 48 50 58 84...


In [57]:
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=config.random_seed,
    stratify=df["emotion"]
)

print(len(train_df))
print(len(val_df))

22967
5742


In [58]:
class_counts = train_df["emotion"].value_counts().sort_index()

print(class_counts)

emotion
0    3196
1     349
2    3277
3    5772
4    3864
5    2537
6    3972
Name: count, dtype: int64


In [59]:
class_weights = 1.0 / torch.tensor(
    class_counts.values,
    dtype=torch.float32
)

class_weights = class_weights / class_weights.sum()

print(class_weights)
class_weights = class_weights.to(device)

tensor([0.0686, 0.6282, 0.0669, 0.0380, 0.0567, 0.0864, 0.0552])


In [60]:
def pixels_to_image(pixel_string):
    pixels = np.fromstring(
        pixel_string,
        dtype=np.uint8,
        sep=" "
    )

    return pixels.reshape(48, 48)

In [61]:
train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ToTensor()
])

In [62]:
class FERDataset(Dataset):

    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform
    

    def __len__(self):
        return len(self.dataframe)


    def __getitem__(self, idx):

        row = self.dataframe.iloc[idx]

        image = pixels_to_image(row["pixels"])

        image = image.astype(np.uint8)


        if self.transform:
            image = self.transform(image)

        else:
            image = image.astype(np.float32) / 255.0
            image = torch.tensor(image).unsqueeze(0)


        label = torch.tensor(
            row["emotion"],
            dtype=torch.long
        )

        return image, label

In [63]:
train_dataset = FERDataset(train_df)
val_dataset = FERDataset(val_df)

In [64]:
images, labels = next(iter(train_loader))

print(images.shape)
print(labels.shape)

torch.Size([64, 1, 48, 48])
torch.Size([64])


In [65]:
import torch
import torch.nn as nn

class SimpleCNN(nn.Module):

    def __init__(self, dropout):
        super().__init__()

        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(16), 
            nn.ReLU(),
            nn.Dropout2d(dropout),
            nn.MaxPool2d(kernel_size=2),

            # Block 2
            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32), 
            nn.ReLU(),
            nn.Dropout2d(dropout),
            nn.MaxPool2d(kernel_size=2),

            # Block 3
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64), 
            nn.ReLU(),
            nn.Dropout2d(dropout),
            nn.MaxPool2d(kernel_size=2)
        )

        self.classifier = nn.Sequential(
            nn.Linear(64 * 6 * 6, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),

            nn.Linear(256, 7)
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, start_dim=1)
        x = self.classifier(x)
        return x

In [66]:
images, labels = next(iter(train_loader))

images = images.to(device)

outputs = model(images)

print("Input shape :", images.shape)
print("Output shape:", outputs.shape)

Input shape : torch.Size([64, 1, 48, 48])
Output shape: torch.Size([64, 7])


In [67]:
def train_epoch(model, loader, criterion, optimizer, device):

    model.train()

    running_loss = 0.0
    correct = 0

    for images, labels in loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad() #?

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()

    epoch_loss = running_loss / len(loader)
    epoch_acc = correct / len(loader.dataset)

    return epoch_loss, epoch_acc

In [68]:
def validate(model, loader, criterion, device):

    model.eval()

    running_loss = 0.0
    correct = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            running_loss += loss.item()

            predictions = outputs.argmax(dim=1)

            correct += (predictions == labels).sum().item()

    epoch_loss = running_loss / len(loader)
    epoch_acc = correct / len(loader.dataset)

    return epoch_loss, epoch_acc

In [69]:
print(next(model.parameters()).device)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

cuda:0
True
Tesla T4


In [70]:
def train():

    run = wandb.init()

    cfg = wandb.config

    model = SimpleCNN(
        dropout=cfg.dropout
    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=cfg.lr
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=cfg.batch_size,
        shuffle=True,
        num_workers=2,
        pin_memory=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=cfg.batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )

    criterion = nn.CrossEntropyLoss(
        weight=class_weights
    )

    best_val_acc = 0

    for epoch in range(config.epochs):

        train_loss, train_acc = train_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
            device
        )

        val_loss, val_acc = validate(
            model,
            val_loader,
            criterion,
            device
        )

        wandb.log({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc
        })

        print(
            f"Epoch {epoch+1}/{config.epochs} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Train Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {val_acc:.4f}"
        )

        if val_acc > best_val_acc:

            best_val_acc = val_acc

            torch.save(
                model.state_dict(),
                "best_baseline_model.pt"
            )

    artifact = wandb.Artifact(
        "best-model",
        type="model"
    )

    artifact.add_file(
        "best_baseline_model.pt"
    )

    wandb.log_artifact(artifact)

    model.eval()

    y_true = []
    y_pred = []

    with torch.no_grad():

        for x, y in val_loader:

            x = x.to(device)

            outputs = model(x)

            preds = outputs.argmax(dim=1)

            y_true.extend(y.numpy())
            y_pred.extend(preds.cpu().numpy())

    report = classification_report(
        y_true,
        y_pred,
        output_dict=True
    )

    wandb.log({
        "precision_macro": report["macro avg"]["precision"],
        "recall_macro": report["macro avg"]["recall"],
        "f1_macro": report["macro avg"]["f1-score"]
    })

    print(
        classification_report(
            y_true,
            y_pred
        )
    )

    run.finish()

In [71]:
sweep_id = wandb.sweep(
    sweep_config,
    project="fer2013"
)

wandb.agent(
    sweep_id,
    function=train,
    count=20
)

Create sweep with ID: g59qiv21
Sweep URL: https://wandb.ai/kende23-n-a/fer2013/sweeps/g59qiv21


wandb: Agent Starting Run: 0uvg3f8t with config:
wandb: 	batch_size: 128
wandb: 	dropout: 0.2
wandb: 	epochs: 20
wandb: 	lr: 0.0005
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


Epoch 1/20 | Train Loss: 1.8085 | Train Acc: 0.2883 | Val Loss: 1.6346 | Val Acc: 0.4053
Epoch 2/20 | Train Loss: 1.6186 | Train Acc: 0.3762 | Val Loss: 1.5247 | Val Acc: 0.4183
Epoch 3/20 | Train Loss: 1.5065 | Train Acc: 0.4206 | Val Loss: 1.4791 | Val Acc: 0.4613
Epoch 4/20 | Train Loss: 1.4301 | Train Acc: 0.4477 | Val Loss: 1.4221 | Val Acc: 0.4410
Epoch 5/20 | Train Loss: 1.3526 | Train Acc: 0.4712 | Val Loss: 1.3761 | Val Acc: 0.4904
Epoch 6/20 | Train Loss: 1.2926 | Train Acc: 0.4934 | Val Loss: 1.3368 | Val Acc: 0.4991
Epoch 7/20 | Train Loss: 1.2336 | Train Acc: 0.5108 | Val Loss: 1.3432 | Val Acc: 0.5051
Epoch 8/20 | Train Loss: 1.1839 | Train Acc: 0.5271 | Val Loss: 1.3286 | Val Acc: 0.4753
Epoch 9/20 | Train Loss: 1.1400 | Train Acc: 0.5470 | Val Loss: 1.2998 | Val Acc: 0.5104
Epoch 10/20 | Train Loss: 1.0943 | Train Acc: 0.5602 | Val Loss: 1.3047 | Val Acc: 0.5145
Epoch 11/20 | Train Loss: 1.0429 | Train Acc: 0.5777 | Val Loss: 1.3253 | Val Acc: 0.5113
Epoch 12/20 | Train

epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
f1_macro,▁
precision_macro,▁
recall_macro,▁
train_acc,▁▃▃▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
train_loss,█▇▆▆▅▅▄▄▄▃▃▃▃▂▂▂▂▁▁▁
val_acc,▁▂▄▃▅▆▆▄▆▆▆▇▇█▇▇▇███
val_loss,█▆▅▄▃▂▂▂▁▁▂▁▂▁▂▂▂▂▁▃
epoch,20
f1_macro,0.51499
precision_macro,0.51265


wandb: Agent Starting Run: 41oag18s with config:
wandb: 	batch_size: 32
wandb: 	dropout: 0.5
wandb: 	epochs: 20
wandb: 	lr: 0.0005
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


...

Epoch 1/20 | Train Loss: 1.7863 | Train Acc: 0.3020 | Val Loss: 1.6250 | Val Acc: 0.3570
Epoch 2/20 | Train Loss: 1.5957 | Train Acc: 0.3856 | Val Loss: 1.5038 | Val Acc: 0.4143
Epoch 3/20 | Train Loss: 1.4901 | Train Acc: 0.4295 | Val Loss: 1.4262 | Val Acc: 0.4514
Epoch 4/20 | Train Loss: 1.4045 | Train Acc: 0.4594 | Val Loss: 1.3808 | Val Acc: 0.4768
Epoch 5/20 | Train Loss: 1.3342 | Train Acc: 0.4833 | Val Loss: 1.3722 | Val Acc: 0.4960
Epoch 6/20 | Train Loss: 1.2649 | Train Acc: 0.5023 | Val Loss: 1.3331 | Val Acc: 0.5051
Epoch 7/20 | Train Loss: 1.2230 | Train Acc: 0.5209 | Val Loss: 1.3086 | Val Acc: 0.4979
Epoch 8/20 | Train Loss: 1.1591 | Train Acc: 0.5389 | Val Loss: 1.3063 | Val Acc: 0.5091
Epoch 9/20 | Train Loss: 1.1241 | Train Acc: 0.5517 | Val Loss: 1.3017 | Val Acc: 0.5136
Epoch 10/20 | Train Loss: 1.0779 | Train Acc: 0.5740 | Val Loss: 1.2863 | Val Acc: 0.5237
Epoch 11/20 | Train Loss: 1.0248 | Train Acc: 0.5861 | Val Loss: 1.2947 | Val Acc: 0.5259
Epoch 12/20 | Train

epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
f1_macro,▁
precision_macro,▁
recall_macro,▁
train_acc,▁▂▃▄▄▄▅▅▅▆▆▆▆▇▇▇▇███
train_loss,█▇▆▆▅▅▄▄▄▃▃▃▃▂▂▂▂▁▁▁
val_acc,▁▃▄▅▆▆▆▆▆▇▇▇▇▇██▇███
val_loss,█▆▄▃▃▂▂▂▁▁▁▁▁▁▁▃▂▃▃▃
epoch,20
f1_macro,0.52639
precision_macro,0.51388


wandb: Agent Starting Run: 2i2lab6b with config:
wandb: 	batch_size: 128
wandb: 	dropout: 0.2
wandb: 	epochs: 20
wandb: 	lr: 0.01
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


Epoch 1/20 | Train Loss: 1.9035 | Train Acc: 0.2346 | Val Loss: 1.7481 | Val Acc: 0.3703
Epoch 2/20 | Train Loss: 1.6677 | Train Acc: 0.3537 | Val Loss: 1.6349 | Val Acc: 0.3960
Epoch 3/20 | Train Loss: 1.5414 | Train Acc: 0.4046 | Val Loss: 1.4653 | Val Acc: 0.4429
Epoch 4/20 | Train Loss: 1.4357 | Train Acc: 0.4416 | Val Loss: 1.3940 | Val Acc: 0.4431
Epoch 5/20 | Train Loss: 1.3468 | Train Acc: 0.4707 | Val Loss: 1.3855 | Val Acc: 0.4720
Epoch 6/20 | Train Loss: 1.2733 | Train Acc: 0.4959 | Val Loss: 1.3504 | Val Acc: 0.4612
Epoch 7/20 | Train Loss: 1.1921 | Train Acc: 0.5261 | Val Loss: 1.3196 | Val Acc: 0.5120
Epoch 8/20 | Train Loss: 1.1046 | Train Acc: 0.5519 | Val Loss: 1.3307 | Val Acc: 0.4920
Epoch 9/20 | Train Loss: 1.0581 | Train Acc: 0.5702 | Val Loss: 1.2790 | Val Acc: 0.5169
Epoch 10/20 | Train Loss: 0.9739 | Train Acc: 0.6044 | Val Loss: 1.2903 | Val Acc: 0.5197
Epoch 11/20 | Train Loss: 0.9088 | Train Acc: 0.6251 | Val Loss: 1.3173 | Val Acc: 0.5352
Epoch 12/20 | Train

epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
f1_macro,▁
precision_macro,▁
recall_macro,▁
train_acc,▁▂▃▄▄▄▅▅▅▅▆▆▆▇▇▇▇███
train_loss,█▇▆▆▅▅▅▄▄▃▃▃▃▂▂▂▂▁▁▁
val_acc,▁▂▄▄▅▅▇▆▇▇██▇▇██████
val_loss,▇▆▃▃▂▂▂▂▁▁▂▃▃▃▄▄▅▇▇█
epoch,20
f1_macro,0.5091
precision_macro,0.50046


In [72]:
wandb.finish()